# Smart City Knowledge Agent
### Agentic AI Engineering Bootcamp — Agentic Extension

This notebook extends the Smart City RAG project into a **tool-using AI agent**.

Instead of always running retrieval, the model can now decide whether it needs the Smart City knowledge tool. When it calls the tool, LangGraph executes the retrieval step, returns evidence to the model, and the model uses that evidence to produce the final response.

## Agentic AI concepts demonstrated

- **Goal-driven behavior** — the agent receives a user request and decides how to handle it.
- **Tool calling** — the LLM can call a Smart City retrieval tool.
- **Dynamic routing** — LangGraph routes execution to the tool only when a tool call is requested.
- **RAG as a tool** — FAISS retrieval becomes an external capability available to the agent.
- **Evidence grounding** — Smart City factual answers are grounded in retrieved chunks.
- **Stateful workflow** — LangGraph maintains the message state through the agent/tool loop.
- **Short-term memory** — an in-memory checkpointer allows multi-turn conversations within a thread.

> This notebook is intentionally separate from `01_smart_city_rag.ipynb`. Notebook 01 demonstrates the RAG foundation; this notebook demonstrates how that RAG capability can be used inside an agentic workflow.


## Architecture

```text
                        User Request
                             │
                             ▼
                       Agent / LLM
                             │
                 ┌───────────┴───────────┐
                 │                       │
        Knowledge needed?          No tool needed
                 │                       │
                Yes                      │
                 ▼                       │
       smart_city_search Tool            │
                 │                       │
                 ▼                       │
          FAISS Retriever                │
                 │                       │
                 ▼                       │
        Top 6 Relevant Chunks            │
                 │                       │
                 └───────────┬───────────┘
                             ▼
                       Agent / LLM
                             │
                             ▼
                         Response
```

The LangGraph loop is:

```text
START → agent → tools → agent → END
                 ▲        │
                 └────────┘
```

The `tools` path is taken only when the model produces a tool call.


## 1. Install Dependencies

In [1]:
%pip install -q langchain langchain-community langchain-openai langchain-classic langgraph faiss-cpu tiktoken beautifulsoup4 pandas

## 2. Configure the OpenAI API Key

In [3]:
import os
from getpass import getpass

if not os.getenv("OPENAI_API_KEY"):
    try:
        from google.colab import userdata
        api_key = userdata.get("OPENAI_API_KEY")  # works only in the Colab browser UI
    except Exception:
        api_key = None
    if not api_key:
        api_key = getpass("Paste your OpenAI API key: ")
    os.environ["OPENAI_API_KEY"] = api_key.strip()

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY was not set.")

# Helps identify requests made by WebBaseLoader.
os.environ.setdefault("USER_AGENT", "smart-city-rag-bootcamp/1.0")

print("OpenAI API key is configured.")



OpenAI API key is configured.


## 3. Build the Smart City Knowledge Base

The agent will use the same type of RAG knowledge base as the first notebook. Six public Smart City webpages are loaded, chunked, embedded, and stored in FAISS.

The agent itself will not directly access these documents. Instead, retrieval will be exposed through a **tool**.


In [4]:
SOURCES = [
    {
        "name": "IBM — What is a Smart City?",
        "url": "https://www.ibm.com/think/topics/smart-city",
        "focus": "Smart-city concepts, AI, IoT, transport, energy, and infrastructure",
    },
    {
        "name": "Riyadh Municipality — Smart City Initiatives",
        "url": "https://alriyadh.gov.sa/en/content/smart-cities-initiatives",
        "focus": "Riyadh smart-city initiatives and municipal services",
    },
    {
        "name": "ITU — Smart Sustainable Cities",
        "url": "https://www.itu.int/en/mediacentre/backgrounders/Pages/smart-sustainable-cities.aspx",
        "focus": "ICT, sustainability, infrastructure, and smart-city development",
    },
    {
        "name": "World Bank — Smart Technology for Sustainable Development",
        "url": "https://www.worldbank.org/en/news/feature/2021/01/18/harnessing-smart-technology-for-sustainable-development-in-developing-countries-tdlc",
        "focus": "Smart technology and sustainable urban development",
    },
    {
        "name": "NIST — CPS / IoT for Smart Cities",
        "url": "https://www.nist.gov/programs-projects/cyber-physical-systemsinternet-things-smart-cities",
        "focus": "IoT, cyber-physical systems, standards, and city services",
    },
    {
        "name": "OECD — Smart Cities and Inclusive Growth",
        "url": "https://www.oecd.org/en/about/programmes/the-oecd-programme-on-smart-cities-and-inclusive-growth0.html",
        "focus": "Well-being, inclusion, public services, sustainability, and digitalisation",
    },
]

print("Configured sources:", len(SOURCES))
for i, source in enumerate(SOURCES, 1):
    print(f"{i}. {source['name']}")


Configured sources: 6
1. IBM — What is a Smart City?
2. Riyadh Municipality — Smart City Initiatives
3. ITU — Smart Sustainable Cities
4. World Bank — Smart Technology for Sustainable Development
5. NIST — CPS / IoT for Smart Cities
6. OECD — Smart Cities and Inclusive Growth


### Load the six webpages

In [5]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.documents import Document

documents = []
load_failures = []

for source in SOURCES:
    try:
        loaded = WebBaseLoader(source["url"]).load()

        if not loaded:
            raise ValueError("No page content was returned.")

        combined_text = "\n\n".join(doc.page_content for doc in loaded)

        metadata = dict(loaded[0].metadata)
        metadata.update(
            {
                "source": source["url"],
                "source_name": source["name"],
                "focus": source["focus"],
            }
        )

        documents.append(
            Document(
                page_content=combined_text,
                metadata=metadata,
            )
        )

    except Exception as exc:
        load_failures.append((source["name"], str(exc)))

print(f"Successfully loaded: {len(documents)} documents")

if load_failures:
    print("\nLoad failures:")
    for name, error in load_failures:
        print(f"- {name}: {error}")


/tmp/ipykernel_2182/667712874.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader


Successfully loaded: 6 documents


In [6]:
assert len(documents) == 6, (
    f"Expected 6 documents, but loaded {len(documents)}. "
    "Review the load failures above."
)

print("Knowledge-base requirement satisfied: 6 documents loaded.")


Knowledge-base requirement satisfied: 6 documents loaded.


### Chunk the documents

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
)

chunks = text_splitter.split_documents(documents)

print("Documents:", len(documents))
print("Chunks:   ", len(chunks))

assert len(chunks) >= 6


Documents: 6
Chunks:    166


### Create embeddings and FAISS

In [8]:
from langchain_openai import OpenAIEmbeddings
from langchain_classic.embeddings import CacheBackedEmbeddings
from langchain_classic.storage import LocalFileStore
from langchain_community.vectorstores import FAISS

CACHE_PATH = "./cache/"
os.makedirs(CACHE_PATH, exist_ok=True)

store = LocalFileStore(CACHE_PATH)

core_embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

embedder = CacheBackedEmbeddings.from_bytes_store(
    core_embeddings_model,
    store,
    namespace=core_embeddings_model.model,
)

vectorstore = FAISS.from_documents(
    chunks,
    embedder,
)

print("FAISS vector store created.")
print("Vectors stored:", vectorstore.index.ntotal)


/usr/local/lib/python3.13/dist-packages/langchain_classic/embeddings/cache.py:58: UserWarning: Using default key encoder: SHA-1 is *not* collision-resistant. While acceptable for most cache scenarios, a motivated attacker can craft two different payloads that map to the same cache key. If that risk matters in your environment, supply a stronger encoder (e.g. SHA-256 or BLAKE2) via the `key_encoder` argument. If you change the key encoder, consider also creating a new cache, to avoid (the potential for) collisions with existing keys.
  _warn_about_sha1_encoder()


FAISS vector store created.
Vectors stored: 166


### Create a top-6 retriever

In [10]:
TOP_K = 6

retriever = vectorstore.as_retriever(
    search_kwargs={"k": TOP_K}
)

print(f"Retriever configured to return top {TOP_K} chunks.")


Retriever configured to return top 6 chunks.


## 4. Turn RAG into an Agent Tool

This is the main transition from the first notebook.

In notebook 01, the application called the retriever directly.

Here, retrieval is wrapped in a LangChain `@tool`. The LLM does not automatically receive the Smart City context. It must **choose to call the tool** when the question requires knowledge from the Smart City knowledge base.


In [11]:
from langchain_core.tools import tool

@tool
def smart_city_search(query: str) -> str:
    """Search the curated Smart City knowledge base.

    Use this tool for factual questions about smart cities, Riyadh smart-city
    initiatives, IoT, AI in cities, smart infrastructure, sustainability,
    urban services, or inclusive smart-city development.

    The tool retrieves the six most relevant chunks from the FAISS knowledge base.
    """

    retrieved_docs = retriever.invoke(query)

    if not retrieved_docs:
        return "No relevant Smart City evidence was found in the knowledge base."

    sections = []

    for rank, doc in enumerate(retrieved_docs, 1):
        source_name = doc.metadata.get("source_name", "Unknown source")
        source_url = doc.metadata.get("source", "Unknown URL")

        sections.append(
            f"""[Retrieved Chunk {rank}]
Source: {source_name}
URL: {source_url}
Content:
{doc.page_content}
"""
        )

    return "\n\n".join(sections)


tools = [smart_city_search]

print("Agent tool created:", smart_city_search.name)


Agent tool created: smart_city_search


### Test the tool independently

Before giving the tool to an agent, verify that the tool itself returns retrieval evidence.


In [12]:
tool_test = smart_city_search.invoke(
    {"query": "How are AI and IoT used in smart cities?"}
)

print(tool_test[:3000])


[Retrieved Chunk 1]
Source: IBM — What is a Smart City?
URL: https://www.ibm.com/think/topics/smart-city
Content:
Artificial intelligence (AI)


Artificial intelligence combines computer science and robust data sets to enable problem-solving. Smart city projects use AI and machine learning-based solutions to manage infrastructure efficiently and sustainably. For example, AI algorithms can optimize waste collection routes, reducing carbon emissions by city garbage trucks. AI can also help law enforcement improve public safety by analyzing data from security cameras and connected devices to detect crime.


[Retrieved Chunk 2]
Source: ITU — Smart Sustainable Cities
URL: https://www.itu.int/en/mediacentre/backgrounders/Pages/smart-sustainable-cities.aspx
Content:
Digital twins and the citiverse: Immersive virtual environments and simulations support city planning, predictive maintenance, and operational efficiency, offering cities the ability to innovate in a risk-free digital space.
Artif

## 5. Configure the Tool-Calling LLM

The model is given access to `smart_city_search`.

The system instructions establish simple grounding rules:

- Use the retrieval tool for factual Smart City questions.
- Do not invent evidence that is not in retrieved context.
- If the knowledge base is insufficient, say so.
- Simple greetings or questions about the agent itself do not require retrieval.


In [14]:
from langchain_openai import ChatOpenAI

base_llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0.0,
)

llm_with_tools = base_llm.bind_tools(tools)

SYSTEM_PROMPT = """
You are a Smart City knowledge agent.

Your goal is to answer user questions accurately while deciding when external
knowledge retrieval is necessary.

TOOL POLICY
1. For factual questions about smart cities, Riyadh smart-city initiatives,
   IoT, AI in cities, smart infrastructure, sustainability, urban services,
   or inclusive smart-city development, use the smart_city_search tool.
2. For simple greetings, conversational messages, or questions about what you
   can do, answer directly without calling the tool.
3. When you use smart_city_search, ground your factual answer in the returned
   evidence.
4. Do not invent facts that are not supported by the retrieved evidence.
5. If the retrieved evidence is insufficient, clearly say that the current
   knowledge base does not contain enough information.
6. When useful, mention the source organizations used in the retrieved context.

Be concise, factual, and transparent about the evidence you used.
""".strip()

print("Tool-calling LLM configured.")


Tool-calling LLM configured.


## 6. Build the LangGraph Agent

`MessagesState` stores the conversation.

The **agent node** calls the LLM. If the LLM requests a tool, `tools_condition` routes execution to the **ToolNode**. After the tool executes, control returns to the agent so the model can interpret the tool result and generate the final response.


In [16]:
from langchain_core.messages import SystemMessage
from langgraph.graph import MessagesState, StateGraph, START
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import InMemorySaver


def call_agent(state: MessagesState):
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        *state["messages"],
    ]

    response = llm_with_tools.invoke(messages)

    return {"messages": [response]}


tool_node = ToolNode(tools)

builder = StateGraph(MessagesState)

builder.add_node("agent", call_agent)
builder.add_node("tools", tool_node)

builder.add_edge(START, "agent")

builder.add_conditional_edges(
    "agent",
    tools_condition,
)

builder.add_edge("tools", "agent")

memory = InMemorySaver()

agent = builder.compile(
    checkpointer=memory
)

print("LangGraph agent compiled.")


LangGraph agent compiled.


### Visualize the graph

In [17]:
from IPython.display import display, Markdown

display(
    Markdown(
        "```mermaid\n"
        + agent.get_graph().draw_mermaid()
        + "\n```"
    )
)


```mermaid
---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	agent(agent)
	tools(tools)
	__end__([<p>__end__</p>]):::last
	__start__ --> agent;
	agent -.-> __end__;
	agent -.-> tools;
	tools --> agent;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

```

## 7. Create an Agent Runner with Trace Inspection

The helper below runs one user query and shows the important execution evidence:

- Did the model request a tool?
- Which tool was requested?
- What query was sent to the tool?
- How many tool calls occurred?
- What final answer did the agent produce?

This makes the agentic behavior observable in the notebook.


In [18]:
from langchain_core.messages import HumanMessage, AIMessage, ToolMessage


def run_agent(query: str, thread_id: str):
    result = agent.invoke(
        {
            "messages": [
                HumanMessage(content=query)
            ]
        },
        config={
            "configurable": {
                "thread_id": thread_id
            }
        },
    )

    new_messages = result["messages"]

    tool_calls = []

    print("=" * 100)
    print("USER")
    print(query)
    print("=" * 100)

    for message in new_messages:

        if isinstance(message, AIMessage) and message.tool_calls:
            for tool_call in message.tool_calls:
                tool_calls.append(tool_call)

                print("\nAGENT DECISION → TOOL CALL")
                print("Tool:", tool_call["name"])
                print("Arguments:", tool_call["args"])

        elif isinstance(message, ToolMessage):
            print("\nTOOL RESULT")
            print(message.content[:1500])

    final_answer = new_messages[-1].content

    print("\nFINAL ANSWER")
    print("-" * 100)
    print(final_answer)

    print("\nTRACE SUMMARY")
    print("-" * 100)
    print("Tool calls:", len(tool_calls))

    return {
        "query": query,
        "tool_calls": tool_calls,
        "tool_used": len(tool_calls) > 0,
        "final_answer": final_answer,
        "messages": new_messages,
    }


## 8. Agent Test 1 — General Smart City Knowledge

This is a factual Smart City question, so the expected behavior is:

```text
User → Agent → smart_city_search → FAISS → Agent → Answer
```


In [19]:
result_1 = run_agent(
    query="What technologies are commonly used in smart cities and how do they improve urban services?",
    thread_id="test-general-smart-city",
)


USER
What technologies are commonly used in smart cities and how do they improve urban services?

AGENT DECISION → TOOL CALL
Tool: smart_city_search
Arguments: {'query': 'technologies commonly used in smart cities and their impact on urban services'}

TOOL RESULT
[Retrieved Chunk 1]
Source: ITU — Smart Sustainable Cities
URL: https://www.itu.int/en/mediacentre/backgrounders/Pages/smart-sustainable-cities.aspx
Content:
TRANSFORMATIVE SOLUTIONS FOR SMARTER, SUSTAINABLE CITIESDigital technologies offer powerful tools to address the multifaceted challenges faced by cities, paving the way for smarter, more efficient, and sustainable urban ecosystems. Key innovations include:
Internet of things (IoT): Connecting devices across urban systems enables real-time monitoring and optimization of services such as traffic management, energy distribution, and public safety.


[Retrieved Chunk 2]
Source: IBM — What is a Smart City?
URL: https://www.ibm.com/think/topics/smart-city
Content:
Technologies 

## 9. Agent Test 2 — Riyadh Smart City

This query requires knowledge about a specific city's initiatives. The agent should call the retrieval tool and use the Riyadh evidence returned by FAISS.


In [20]:
result_2 = run_agent(
    query="How is Riyadh using AI and IoT to improve municipal services?",
    thread_id="test-riyadh",
)


USER
How is Riyadh using AI and IoT to improve municipal services?

AGENT DECISION → TOOL CALL
Tool: smart_city_search
Arguments: {'query': 'Riyadh use of AI and IoT to improve municipal services'}

TOOL RESULT
[Retrieved Chunk 1]
Source: Riyadh Municipality — Smart City Initiatives
URL: https://alriyadh.gov.sa/en/content/smart-cities-initiatives
Content:
through sensors to determine the used, the non-used, and the problems that occur to them. These initiatives reflect Riyadh Municipality's vision of transforming the city into a global model for smart cities, while continuing to develop innovative solutions to improve quality of life and promote sustainability.  The Riyadh Municipality is striving to achieve a comprehensive digital transformation that supports the development of a smart city through several initiatives aligned with the


[Retrieved Chunk 2]
Source: Riyadh Municipality — Smart City Initiatives
URL: https://alriyadh.gov.sa/en/content/smart-cities-initiatives
Content:
Dro

## 10. Agent Test 3 — No Retrieval Needed

This test demonstrates **dynamic tool choice**.

A greeting does not require external Smart City evidence, so the expected path is:

```text
User → Agent → Answer
```

No FAISS retrieval should occur.


In [21]:
result_3 = run_agent(
    query="Hello! What can you help me with?",
    thread_id="test-no-tool",
)


USER
Hello! What can you help me with?

FINAL ANSWER
----------------------------------------------------------------------------------------------------
Hello! I can help you with information and questions about smart cities, including topics like Riyadh's smart-city initiatives, IoT and AI applications in urban environments, smart infrastructure, sustainability, urban services, and inclusive smart-city development. Feel free to ask me anything related to these areas!

TRACE SUMMARY
----------------------------------------------------------------------------------------------------
Tool calls: 0


## 11. Validate the Agent's Tool Decisions

The expected behavior is:

| Test | Expected |
|---|---|
| Smart City factual question | Tool used |
| Riyadh factual question | Tool used |
| Greeting | No tool |


In [22]:
import pandas as pd

decision_results = pd.DataFrame(
    [
        {
            "test": "General Smart City question",
            "expected_tool_use": True,
            "actual_tool_use": result_1["tool_used"],
        },
        {
            "test": "Riyadh Smart City question",
            "expected_tool_use": True,
            "actual_tool_use": result_2["tool_used"],
        },
        {
            "test": "Greeting",
            "expected_tool_use": False,
            "actual_tool_use": result_3["tool_used"],
        },
    ]
)

decision_results["status"] = decision_results.apply(
    lambda row: (
        "PASS"
        if row["expected_tool_use"] == row["actual_tool_use"]
        else "REVIEW"
    ),
    axis=1,
)

decision_results


,test,expected_tool_use,actual_tool_use,status
0,General Smart City question,True,True,PASS
1,Riyadh Smart City question,True,True,PASS
2,Greeting,False,False,PASS


In [24]:
assert result_1["tool_used"], (
    "Expected the agent to retrieve Smart City knowledge for Test 1."
)

assert result_2["tool_used"], (
    "Expected the agent to retrieve Riyadh knowledge for Test 2."
)

assert not result_3["tool_used"], (
    "Expected the agent to answer the greeting without retrieval."
)

print("Agent tool-routing checks passed.")


Agent tool-routing checks passed.


## 12. Demonstrate Short-Term Conversation Memory

Because the graph was compiled with an `InMemorySaver`, messages can persist within the same `thread_id`.

The second question below intentionally refers to the first answer with **“those technologies.”** The agent receives the previous conversation state and can interpret the follow-up in context.


In [25]:
memory_thread = "smart-city-memory-demo"

memory_result_1 = run_agent(
    query="Explain how IoT is used in smart cities.",
    thread_id=memory_thread,
)

memory_result_2 = run_agent(
    query="Which of those technologies are relevant to Riyadh's city services?",
    thread_id=memory_thread,
)


USER
Explain how IoT is used in smart cities.

AGENT DECISION → TOOL CALL
Tool: smart_city_search
Arguments: {'query': 'IoT usage in smart cities'}

TOOL RESULT
[Retrieved Chunk 1]
Source: IBM — What is a Smart City?
URL: https://www.ibm.com/think/topics/smart-city
Content:
Areas of city operations where ICT, IoT and other smart technologies increasingly play an important role include transportation, energy and infrastructure. As a city updates its systems and structures to incorporate these technologies, it becomes smarter. However, exactly which cities should be considered smart cities or should claim the mantle of the “smartest” city is a matter of debate.


[Retrieved Chunk 2]
Source: ITU — Smart Sustainable Cities
URL: https://www.itu.int/en/mediacentre/backgrounders/Pages/smart-sustainable-cities.aspx
Content:
TRANSFORMATIVE SOLUTIONS FOR SMARTER, SUSTAINABLE CITIESDigital technologies offer powerful tools to address the multifaceted challenges faced by cities, paving the way for

## 13. Inspect the Stored Conversation State

This shows that the workflow can maintain state across turns instead of treating every prompt as an isolated request.


In [26]:
state = agent.get_state(
    {
        "configurable": {
            "thread_id": memory_thread
        }
    }
)

print("Messages stored in this thread:", len(state.values["messages"]))

for i, message in enumerate(state.values["messages"], 1):
    print(
        f"{i}. {type(message).__name__}: "
        f"{str(message.content)[:160]}"
    )


Messages stored in this thread: 8
1. HumanMessage: Explain how IoT is used in smart cities.
2. AIMessage: 
3. ToolMessage: [Retrieved Chunk 1]
Source: IBM — What is a Smart City?
URL: https://www.ibm.com/think/topics/smart-city
Content:
Areas of city operations where ICT, IoT and ot
4. AIMessage: IoT is used in smart cities by connecting devices across urban systems to enable real-time monitoring and optimization of services such as traffic management, e
5. HumanMessage: Which of those technologies are relevant to Riyadh's city services?
6. AIMessage: 
7. ToolMessage: [Retrieved Chunk 1]
Source: Riyadh Municipality — Smart City Initiatives
URL: https://alriyadh.gov.sa/en/content/smart-cities-initiatives
Content:
through senso
8. AIMessage: In Riyadh's city services, relevant smart city technologies include the use of sensors and intelligent systems for real-time data analysis, such as in the smart


## 14. What Makes This Agentic?

The first notebook followed a fixed RAG pipeline:

```text
Query → Retrieve → Generate
```

This notebook introduces a decision-making loop:

```text
                  ┌───────────────┐
                  │     Agent     │
                  └───────┬───────┘
                          │
                   Need a tool?
                    /          \
                  Yes           No
                   │             │
                   ▼             ▼
            Retrieval Tool    Respond
                   │
                   ▼
              Tool Result
                   │
                   └──────► Agent
```


### Concepts demonstrated

- LLM tool calling
- ReAct-style agent/tool loop
- LangGraph orchestration
- Conditional routing
- RAG as an agent tool
- State management
- Short-term conversation memory
- Evidence-grounded responses
- Observable tool decisions
